# 01 - 读取 IDX 文件 & 数据预处理

## 目标
- 用 `open(..., 'rb')` 读取 MNIST 原始 IDX 格式
- 理解二进制 header：magic number → 维度 → 数据
- reshape：784 ↔ 28×28
- normalize：`x / 255.0`
- one-hot 编码标签
- matplotlib 可视化几张图片

## IDX 格式
| 偏移（字节） | 内容 |
|---|---|
| 0–3 | magic number（大端序） |
| 4–7 | 图片数量 |
| 8–11 | 行数（28） |
| 12–15 | 列数（28） |
| 16+ | 像素数据 |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import struct

"""
with ... as f: 是上下文管理器，进入block时打开文件，离开block时自动关闭文件，本质安全开关文件。后一个f会覆盖前面的
rb read binary
struct.unpack() 将二进制数据解包成 Python 对象，unpack是指按指定格式解析，返回一个元组：(2051, 60000, 28, 28)
> 表示大端序(big-endian)
I 表示 4 字节无符号整数
'>IIII' = 一次连续读 4 个整数，总共 16 字节
f.read(16) 从文件中读取前 16 字节数据，返回一个 bytes 对象
文件对象内部维护了一个位置指针，指向"下一次读取从哪里开始"，f.read返回bytes对象
np.frombuffer(f.read(), dtype=np.uint8)：从bytes对象中读取数据并转换为numpy数组
uint8：因为是灰度图像，每个像素值在0-255之间，使用8位无符号整数表示
不用的值用 _ 接，不存变量名
"""



# 读取训练图片
with open('../datasets/train-images.idx3-ubyte', 'rb') as f:
    _, train_num, rows, cols = struct.unpack('>IIII', f.read(16))
    images = np.frombuffer(f.read(), dtype=np.uint8).reshape(train_num, rows, cols)

# 读取测试图片
with open("../datasets/t10k-images.idx3-ubyte", "rb") as f:
    _, test_num, _, _ = struct.unpack(">IIII", f.read(16))
    test_images = np.frombuffer(f.read(), dtype=np.uint8).reshape(test_num, rows, cols)

# 读取训练标签
with open("../datasets/train-labels.idx1-ubyte", "rb") as f:
    f.read(8)  # 跳过 header
    labels = np.frombuffer(f.read(), dtype=np.uint8)

# 读取测试标签
with open("../datasets/t10k-labels.idx1-ubyte", "rb") as f:
    f.read(8)
    test_labels = np.frombuffer(f.read(), dtype=np.uint8)


# print(f"训练集: {len(images)} 张图片, {len(labels)} 个标签")
# print(f"测试集: {len(test_images)} 张图片, {len(test_labels)} 个标签")
# print(f"图片尺寸: {rows}x{cols}")


# 可视化前10张图片和标签
# fig, axes = plt.subplots(2, 5, figsize=(10, 5))
# for i, ax in enumerate(axes.flat):
#     ax.imshow(images[i], cmap='gray')
#     ax.set_title(f'label: {labels[i]}')
#     ax.axis('off')
# plt.tight_layout()
# plt.show()


"""
由于sigmoid的饱和特性，以及导数在输入较大或较小时趋近于0，导致梯度消失问题，训练困难。将像素值归一化到[0,1]范围内
归一化后优化也更容易
float32的两个原因：
  1. 除法会出小数。uint8 只能存 0~255 的整数，200 / 255 = 0.784...，用 uint8 放不下，直接给你截成 0。
  2. 后续计算需要梯度。权重是小数（0.01、-0.5 这种），矩阵乘法 x @ W 如果 x是整数类型，结果会被截断，梯度更新也会废掉。
  3. 选 float32 而不是 float64：训练时要处理 60000×784 的矩阵，float32 比 float64 省一半内存，而且MNIST这种任务 float32 精度完全够用。

最简单的全连接网络(每个输入都连接到下一层所有神经元)(MLP多层感知机)只能处理vector，但会丢失空间结构信息
约定x代表输入(特征)，y代表标签(正确答案)，W代表权重，b代表偏置
只要训练和测试用同一种方式展平，就没问题。numpy 默认 C order
"""
# 归一化 + 展平
x_train = images.astype(np.float32) / 255.0       # [0,255] → [0,1]
x_train = x_train.reshape(images.shape[0], -1)     # (60000,28,28) → (60000,784)

x_test = test_images.astype(np.float32) / 255.0
x_test = x_test.reshape(test_images.shape[0], -1)  # (10000,28,28) → (10000,784)


"""
one-hot 把"这个样本属于哪类"表达成一个概率分布——正确类的概率是 1，其余是 0。表示的是正确答案是什么
向量里只有一个位置是"热"的（1），其余都是"冷"的（0）
后面与 softmax 结合使用，softmax 输出一个概率分布，one-hot 是正确答案的概率分布，两者一起计算交叉熵损失
"""

# 将标签转换为 one-hot 编码
def to_onehot(labels, num_classes=10):
    onehot = np.zeros((len(labels), num_classes))       #按行摆放onehot vector
    onehot[np.arange(len(labels)), labels] = 1.0
    return onehot

y_train = to_onehot(labels)
y_test = to_onehot(test_labels)

# print(f"x_train: {x_train.shape}")   # (60000, 784)
# print(f"y_train: {y_train.shape}")   # (60000, 10)
# print(f"x_test:  {x_test.shape}")    # (10000, 784)
# print(f"y_test:  {y_test.shape}")    # (10000, 10)